# Kaspix Dataset Generator — Gemelo Digital AD8572

**Componente:** Analog Devices AD8572 (Dual Precision Zero-Drift Op-Amp, Rail-to-Rail I/O)  
**Circuito modelado:** Amplificador inversor de precisión con filtro RC de entrada  
**Knobs parametrizables:** `Rf` (ganancia), `Rg` (ganancia), `Rin` (impedancia entrada), `Cf` (polo de estabilidad), `Vcc` (alimentación)  
**Pipeline:** Kaspix Omni-Pipeline V4.2 — adaptado AD8572

---
### Características físicas del AD8572 (para referencia del netlist)
- **GBW:** ~1.5 MHz  
- **Slew Rate:** 0.4 V/µs  
- **Alimentación:** 2.7 V – 5.5 V (single) / ±2.75 V (dual)  
- **Vos:** < 1 µV (autozeroing)  
- **Corriente de bias:** 10 pA típico  
- **CMRR:** 130 dB típico  
- **Rout (lazo abierto):** ~160 Ω  
- **Ruido en voltaje:** 22 nV/√Hz @ 1 kHz

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

!sudo apt-get install -y ngspice libngspice0

import os
if not os.path.exists('/usr/lib/x86_64-linux-gnu/libngspice.so'):
    os.system('sudo ln -s /usr/lib/x86_64-linux-gnu/libngspice.so.0 /usr/lib/x86_64-linux-gnu/libngspice.so')
    print('Created symlink libngspice.so')

os.environ['PYSPICE_NGSPICE_PATH'] = '/usr/lib/x86_64-linux-gnu/libngspice.so.0'

!pip install PySpice
!dpkg -L ngspice
!ls -l /usr/lib/x86_64-linux-gnu/libngspice.so
!ldd /usr/lib/x86_64-linux-gnu/libngspice.so

import PySpice.Logging.Logging as Logging
import logging
from PySpice.Spice.Netlist import Circuit
from PySpice.Unit import *
from PySpice.Spice.Parser import SpiceParser
import re
import torch
import scipy.signal as signal
from tqdm.auto import tqdm
from torch.utils.data import Dataset, DataLoader
import concurrent.futures
import multiprocessing
import random
import glob
import shutil

logger = Logging.setup_logging()
logger.setLevel(logging.ERROR)
print('✅ Setup completado')

## Netlist NGSpice — AD8572 Amplificador Inversor de Precisión

El AD8572 se modela con un **subcircuito behavioural** que captura:
1. **GBW finito** (~1.5 MHz) modelado con polos dominantes
2. **Slew Rate** (0.4 V/µs) via corriente limitada en condensador interno
3. **Limitación Rail-to-Rail** de salida (Vcc - 50mV)
4. **Impedancia de entrada diferencial** alta (~1 TΩ)
5. **Impedancia de salida** (~160 Ω en lazo abierto)

El circuito completo es un **amplificador inversor** donde:
- `Rf/Rg` fija la ganancia de tensión: `Av = -Rf/Rg`
- `Rin + Cin` forma un filtro paso-bajo en la entrada (anti-aliasing)
- `Cf` en paralelo con `Rf` limita el BW de realimentación (estabilidad)
- `Vcc` controla la alimentación (single supply, mid-rail bias)

**Nota:** El modelo SPICE oficial de ADI está encriptado (`.lib` propietario). Este modelo behavioural open-source es representativo para entrenamiento de gemelos digitales.

In [ ]:
# ESCRITURA DEL NETLIST AD8572 (Behavioural NGSpice)

import os
os.makedirs('circuits_ad8572', exist_ok=True)

AD8572_NETLIST = """\
* AD8572 Precision Inverting Amplifier — Kaspix Behavioural Model

.param Rf=100k
.param Rg=10k
.param Rin=1k
.param Cf=100p
.param Cin=1n

Vcc vcc 0 DC 5.0
Vmid vmid 0 DC 2.5
Vin input 0 DC 0 AC 1

Rin1 input nin {Rin}
Cin1 nin 0 {Cin}
Rg1 nin ninv {Rg}
Rf1 ninv output {Rf}
Cf1 ninv output {Cf}

EOpAmp  output  0  vmid  ninv  100000

* OpAmp sera inyectado por el worker como VCVS
* Xopamp vmid ninv output vcc 0 AD8572A

.tran 1u 0.04
.save v(output)
.end
"""

NETLIST_PATH = 'circuits_ad8572/ad8572_inverting_amp.cir'
with open(NETLIST_PATH, 'w') as f:
    f.write(AD8572_NETLIST)

print(f' Netlist guardado: {NETLIST_PATH}')
print(f'   Knobs: Rf, Rg, Rin, Cf, Cin')
print(f'   Ganancia nominal: -Rf/Rg = -{100000/10000:.0f}x = -{20*np.log10(100000/10000):.1f} dB')
print(f'   Polo entrada: {1/(2*np.pi*1000*1e-9):.0f} Hz')
print(f'   Polo feedback: {1/(2*np.pi*100000*100e-12):.0f} Hz')

## NetlistProcessor — Análisis y Extracción de Parámetros

In [ ]:
file_path = 'circuits_ad8572/ad8572_inverting_amp.cir'
class NetlistProcessor:
    """
    Motor de análisis de Netlists para el Kaspix Omni-Pipeline.
    Adaptado para componentes AD (AD8572).
    """

    def __init__(self, file_path):
        self.file_path = file_path
        self.content = ''
        self.params_raw = {}
        self.params_float = {}
        self.io_config = {
            'input_source': None,
            'output_node': None,
            'ground_check': False
        }
        self.is_parsed = False
        self._load_file()

    def _load_file(self):
        if not os.path.exists(self.file_path):
            raise FileNotFoundError(f'[Kaspix Error] No se encuentra: {self.file_path}')
        with open(self.file_path, 'r', encoding='utf-8') as f:
            AD8572A_SUB = """
            .subckt AD8572A non_inv inv out vdd gnd
            * Parámetros internos
            .param AOL=316227
            .param Rout=160
            * 1. Impedancia diferencial de entrada
            Rin_diff non_inv inv 1T
            * 2. Ganancia AOL con polo dominante (GBW=1.5MHz)
            *    fp_dom = GBW/AOL = 1.5MHz/316227 ≈ 4.7 Hz
            Ea  va gnd non_inv inv {AOL}
            Rp1 va vb 1k
            Cp1 vb gnd 33.8u
            * 3. Polo no-dominante (~3 MHz para margen de fase)
            Rp2 vb vc 1k
            Cp2 vc gnd 53n
            * 4. Buffer de salida con Rout
            Eout vout_int gnd vc gnd 1
            Rout1 vout_int out {Rout}
            .ends AD8572A
            """
        with open(file_path, 'r', encoding='utf-8') as f:
            lines = f.readlines()
        self.content = lines[0] + AD8572A_SUB + "".join(lines[1:])

    def _spice_to_float(self, value_str):
        value_str = value_str.lower().strip('{} ')
        suffixes = {
            't': 1e12, 'g': 1e9, 'meg': 1e6, 'k': 1e3,
            'mil': 25.4e-6, 'm': 1e-3, 'u': 1e-6,
            'n': 1e-9, 'p': 1e-12, 'f': 1e-15
        }
        match = re.match(r'^([\d\.\-\+]+)([a-z]*)$', value_str)
        if not match:
            try:
                return float(value_str)
            except ValueError:
                return None
        number, suffix = match.groups()
        multiplier = 1.0
        if suffix:
            if suffix.startswith('meg'):
                multiplier = suffixes['meg']
            elif suffix[0] in suffixes:
                multiplier = suffixes[suffix[0]]
        return float(number) * multiplier

    def analyze(self):
        lines = self.content.splitlines()
        re_param  = re.compile(r'\.param\s+(\w+)\s*=\s*(\{?[\w\.\-\+]+\}?)', re.IGNORECASE)
        re_source = re.compile(r'^\s*([vV]\w+)\s+(\w+)\s+(\w+)', re.IGNORECASE)
        re_save   = re.compile(r'\.(?:save|print)\s+(?:v|tran)?\(?v?\((.+?)\)\)?', re.IGNORECASE)

        input_source = None
        output_target = None
        knobs_values = {}
        valid_spice = True

        for line in lines:
            line_strip = line.strip()
            if not line_strip or line_strip.startswith('*'):
                continue

            # Parámetros .param
            pm = re_param.search(line_strip)
            if pm:
                name, val_str = pm.groups()
                self.params_raw[name] = val_str
                fval = self._spice_to_float(val_str)
                if fval is not None:
                    self.params_float[name] = fval

            # Fuente de entrada (primera fuente V que no sea Vcc ni Vmid)
            sm = re_source.match(line_strip)
            if sm:
                v_name = sm.group(1)
                v_name_upper = v_name.upper()
                # Excluir fuentes de alimentación y referencia
                exclude = ['VCC', 'VMID', 'VPOS', 'VNEG', 'VDD', 'VSS', 'VREF']
                if v_name_upper not in exclude and input_source is None:
                    input_source = v_name

            # Nodo de salida (.save)
            sv = re_save.search(line_strip)
            if sv and output_target is None:
                output_target = 'v(' + sv.group(1).strip() + ')'

        # Fallbacks seguros
        if input_source is None:
            input_source = 'Vin'
        if output_target is None:
            output_target = 'v(output)'

        # Los knobs son SOLO los parámetros controlables del circuito
        # (excluir parámetros internos del subcircuito)
        KNOB_PARAMS = ['Rf', 'Rg', 'Rin', 'Cf', 'Cin']
        for k in KNOB_PARAMS:
            if k in self.params_float:
                knobs_values[k] = self.params_float[k]

        self.is_parsed = True
        result = {
            'file_path': self.file_path,
            'input_source': input_source,
            'output_target': output_target,
            'knobs_values': knobs_values,
            'all_params': self.params_float,
            'valid_spice': valid_spice
        }
        print(f'[NetlistProcessor] Analizado: {self.file_path}')
        print(f'  Fuente entrada : {input_source}')
        print(f'  Nodo salida    : {output_target}')
        print(f'  Knobs ({len(knobs_values)}): {knobs_values}')
        return result

# Test rápido
proc_test = NetlistProcessor(NETLIST_PATH)
meta_test  = proc_test.analyze()

## Funciones Core — Reproducibilidad, Signal Factory, clean_and_align

In [ ]:
# 0. FUNCIONES AUXILIARES PARA REPRODUCIBILIDAD
def set_global_seed(seed=42):
    import random, numpy as np, torch
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    print(f' Kaspix Seed Locked: {seed}')


# 1. KASPIX SIGNAL FACTORY (Con Dinámica de Amplitud)
class KaspixSignalFactory:
    def __init__(self, fs, duration_sec):
        self.fs = fs
        self.duration = duration_sec
        self.n_samples = int(fs * duration_sec)
        self.time_axis = np.linspace(0, duration_sec, self.n_samples)

    def _apply_dynamic_gain(self, y):
        target_peak = np.exp(np.random.uniform(np.log(0.05), np.log(1.5)))
        max_val = np.max(np.abs(y))
        if max_val > 1e-9:
            return (y / max_val) * target_peak
        return y

    def _apply_input_noise(self, signal_in):
        if np.random.rand() > 0.9:
            return signal_in
        noise = np.random.randn(len(signal_in))
        noise_level_db = np.random.uniform(-70, -30)
        noise_amplitude = 10 ** (noise_level_db / 20)
        noisy_signal = signal_in + (noise * noise_amplitude)
        # AD8572 alimentado a ±2.5V → clip a ±1.5V (entrada útil)
        return np.clip(noisy_signal, -1.5, 1.5)

    def _pink_noise(self):
        uneven = self.n_samples % 2
        X = np.random.randn(self.n_samples // 2 + 1 + uneven) + \
            1j * np.random.randn(self.n_samples // 2 + 1 + uneven)
        S = np.sqrt(np.arange(len(X)) + 1.)
        y = (np.fft.irfft(X / S)).real
        if uneven: y = y[:-1]
        y = self._apply_dynamic_gain(y)
        return self._apply_input_noise(y)

    def _chirp_log(self):
        f_start = 1
        # AD8572 GBW=1.5MHz → chirp hasta ~500kHz
        f_end = np.random.uniform(1e3, min(500e3, self.fs / 2 * 0.9))
        y = signal.chirp(self.time_axis, f0=f_start, f1=f_end,
                         t1=self.duration, method='logarithmic')
        y = self._apply_dynamic_gain(y)
        return self._apply_input_noise(y)

    def _step_sequence(self):
        num_steps = np.random.randint(3, 12)
        y = np.zeros_like(self.time_axis)
        indices = np.sort(np.random.choice(np.arange(self.n_samples), num_steps, replace=False))
        levels = np.random.uniform(-0.8, 0.8, num_steps)
        current_idx = 0
        for i, idx in enumerate(indices):
            y[current_idx:idx] = levels[i-1] if i > 0 else 0
            current_idx = idx
        y[current_idx:] = levels[-1]
        return self._apply_input_noise(y)

    def _multitone(self):
        num_tones = np.random.randint(3, 15)
        y = np.zeros_like(self.time_axis)
        for _ in range(num_tones):
            freq = np.random.uniform(1, min(200e3, self.fs / 3))
            phase = np.random.uniform(0, 2 * np.pi)
            y += np.sin(2 * np.pi * freq * self.time_axis + phase)
        y = self._apply_dynamic_gain(y)
        return self._apply_input_noise(y)

    def _impulse_train(self):
        y = np.zeros_like(self.time_axis)
        num_impulses = np.random.randint(1, 5)
        indices = np.random.randint(0, self.n_samples, num_impulses)
        y[indices] = 1.0
        return self._apply_input_noise(y)

    def _sensor_ramp(self):
        """Señal tipo rampa/sensor — típica en aplicaciones del AD8572."""
        y = np.linspace(-0.5, 0.5, self.n_samples)
        # Añadir variaciones lentas (drift + ruido)
        drift = 0.1 * np.sin(2 * np.pi * 0.5 * self.time_axis)
        y = y + drift
        return self._apply_input_noise(self._apply_dynamic_gain(y))

    def get_signal(self, recipe):
        keys  = list(recipe.keys())
        probs = np.array(list(recipe.values()))
        probs /= probs.sum()
        choice = np.random.choice(keys, p=probs)

        dispatch = {
            'chirp':        (self._chirp_log,      'Chirp Log'),
            'pink_noise':   (self._pink_noise,      'Pink Noise'),
            'step_sequence':(self._step_sequence,   'Step Seq'),
            'multitone':    (self._multitone,        'Multitone'),
            'impulse':      (self._impulse_train,   'Impulse'),
            'sensor_ramp':  (self._sensor_ramp,     'Sensor Ramp'),
            'sine': (
                lambda: self._apply_input_noise(self._apply_dynamic_gain(
                    np.sin(2 * np.pi * np.random.uniform(1, 100e3) * self.time_axis)
                )), 'Sine'
            ),
        }

        if choice in dispatch:
            fn, name = dispatch[choice]
            return fn(), name
        return self._pink_noise(), 'Default (Pink)'


# 2. LIMPIEZA Y ALINEACIÓN
def clean_and_align(audio_in, target):
    """
    Elimina DC Offset y sincroniza señales via correlación cruzada.
    Garantiza Lag=0 para la red TCN.
    """
    from scipy.signal import correlate
    audio_in_zero = audio_in - np.mean(audio_in)
    target_zero   = target   - np.mean(target)

    correlation = correlate(target_zero, audio_in_zero, mode='full')
    lag  = np.argmax(correlation) - (len(audio_in_zero) - 1)

    if lag > 0:
        target_aligned = target_zero[lag:]
        audio_aligned  = audio_in_zero[:-lag]
    elif lag < 0:
        target_aligned = target_zero[:lag]
        audio_aligned  = audio_in_zero[-lag:]
    else:
        target_aligned, audio_aligned = target_zero, audio_in_zero

    return audio_aligned, target_aligned, lag


print('Funciones core cargadas (Signal Factory, clean_and_align, set_global_seed)')

## Simulation Worker — AD8572 adaptado

Cambios vs. filtros activos:
- **Knob sampling** con rangos físicamente válidos para el AD8572
- **Clip de señal** respetando el rango de entrada del op-amp
- **Fix de subcircuito**: el subcircuito `AD8572A` se mantiene (no se sustituye por VCVS genérico)
- **Detección de output** apunta a nodo `output` del circuito inversor

In [ ]:
# SIMULATION WORKER FOR AD8572
def _simulation_worker_ad8572(task_payload):
    import numpy as np
    import re
    from PySpice.Spice.Parser import SpiceParser

    def sanitize_netlist(content):
        content = re.sub(
            r'\{([^{}]*)\}',
            lambda m: '{' + m.group(1).replace(' ', '') + '}',
            content
        )
        lines = content.splitlines()
        if not lines: return content
        title, body, subckts = lines[0], [], []
        in_subckt, buffer = False, []
        for line in lines[1:]:
            l_lower = line.strip().lower()
            if l_lower.startswith('.subckt'):
                in_subckt = True
                buffer.append(line)
            elif l_lower.startswith('.ends'):
                in_subckt = False
                buffer.append(line)
                subckts.extend(buffer)
                buffer = []
            elif in_subckt:
                buffer.append(line)
            else:
                body.append(line)
        return '\n'.join([title] + subckts + body)

    try:
        file_path     = task_payload['file_path']
        input_source  = task_payload['input_source']
        output_target = task_payload['output_target']
        fs            = float(task_payload['fs'])
        duration      = float(task_payload['duration'])
        input_signal  = task_payload['input_signal']
        knob_config   = task_payload['knob_config']
        task_id       = task_payload['id']

        # 1. Leer y sanitizar netlist
        with open(file_path, 'r', encoding='utf-8') as f:
            raw_content = f.read()
        clean_content = sanitize_netlist(raw_content)

        parser  = SpiceParser(source=clean_content)
        circuit = parser.build_circuit()

        # 2. Inyectar señal de entrada PWL
        actual_source_name = None
        for element in circuit.element_names:
            if element.upper() == input_source.upper():
                actual_source_name = element
                break

        if actual_source_name:
            original_source = circuit[actual_source_name]
            n_pos = str(original_source.nodes[0])
            n_neg = str(original_source.nodes[1])
            circuit._elements.pop(actual_source_name)

            time_axis  = np.linspace(0, duration, len(input_signal))
            input_data = list(zip(
                [float(t) for t in time_axis],
                [float(v) for v in input_signal]
            ))
            circuit.PieceWiseLinearVoltageSource(
                'Input_UES', n_pos, n_neg, values=input_data
            )
        else:
            return {'status': 'error',
                    'msg': f"Fuente '{input_source}' no hallada",
                    'id': task_id}

        # 3. Sustituir knobs directamente en el string del netlist
        for name, val in knob_config.items():
            raw_content = re.sub(
                r'\{' + name + r'\}',
                str(float(val)),
                raw_content,
                flags=re.IGNORECASE
            )
        # También reemplazar los .param (para que no queden definiciones huérfanas)
        raw_content = re.sub(
            r'\.param\s+\w+\s*=\s*[\w\.]+\n',
            '',
            raw_content,
            flags=re.IGNORECASE
        )

        clean_content = sanitize_netlist(raw_content)
        parser  = SpiceParser(source=clean_content)
        circuit = parser.build_circuit()


        # 4. Simulación transient
        simulator = circuit.simulator(temperature=25, nominal_temperature=25)
        simulator.options(reltol=0.01, method='gear', abstol='1e-10', itl4=100)

        try:
            step  = float(1.0 / fs)
            end_t = float(duration)
            analysis = simulator.transient(step_time=step, end_time=end_t)
        except Exception as e:
            return {'status': 'error', 'msg': f'NGSpice Crash: {str(e)}', 'id': task_id}

        # 5. Extraer señal de salida
        target_clean = output_target.upper().replace('V(', '').replace(')', '').strip()
        found_node = None
        for node_name in analysis.nodes:
            if str(node_name).upper() == target_clean:
                found_node = node_name
                break
        if found_node is None:
            for node_name in analysis.nodes:
                if 'OUTPUT' in str(node_name).upper():
                    found_node = node_name
                    break
        if found_node is None:
            return {'status': 'error',
                    'msg': f"Salida '{target_clean}' no encontrada. Nodos: {list(analysis.nodes)}",
                    'id': task_id}

        output_signal = np.array(analysis.nodes[found_node])

        # 6. Validaciones
        if np.isnan(output_signal).any():
            return {'status': 'error', 'msg': 'NaN detectado', 'id': task_id}
        if np.max(np.abs(output_signal)) > 200:
            return {'status': 'error', 'msg': 'Explosión numérica (>200V)', 'id': task_id}

        # Reinterpolación si NGSpice devuelve longitud diferente
        time_axis = np.linspace(0, duration, len(input_signal))
        if len(output_signal) != len(time_axis):
            output_signal = np.interp(time_axis, np.array(analysis.time), output_signal)

        return {
            'status': 'ok',
            'id': task_id,
            'y': output_signal.astype(np.float32),
            'x_meta': {
                'knobs': np.array(list(knob_config.values()), dtype=np.float32),
                'knob_names': list(knob_config.keys())
            }
        }

    except Exception as e:
        return {
            'status': 'error',
            'msg': f'Worker Logic Fail: {str(e)}',
            'id': task_payload.get('id', -1)
        }

print(' Worker AD8572 definido')

## KaspixParallelGenerator — AD8572

**Rangos de Knobs para el AD8572:**
| Knob | Nominal | Rango Dataset | Efecto Físico |
|------|---------|--------------|---------------|
| `Rf` | 100 kΩ | 10 kΩ – 1 MΩ | Ganancia inversora |
| `Rg` | 10 kΩ  | 1 kΩ – 100 kΩ | Ganancia inversora |
| `Rin`| 1 kΩ   | 500 Ω – 10 kΩ | Polo de entrada |
| `Cf` | 100 pF | 10 pF – 10 nF | Polo de feedback / estabilidad |
| `Cin`| 1 nF   | 100 pF – 100 nF | BW filtro entrada |

In [ ]:
# KASPIX PARALLEL GENERATOR FOR AD8572

# Rangos físicos válidos para el AD8572 (min, max)
AD8572_KNOB_RANGES = {
    'Rf':  (10e3,   1e6),     # 10 kΩ – 1 MΩ
    'Rg':  (1e3,    100e3),   # 1 kΩ – 100 kΩ
    'Rin': (500,    10e3),    # 500 Ω – 10 kΩ
    'Cf':  (10e-12, 10e-9),   # 10 pF – 10 nF
    'Cin': (100e-12, 100e-9), # 100 pF – 100 nF
}


class KaspixParallelGeneratorAD8572:
    def __init__(self, processor_result, fs=500000, duration_sec=0.04, recipe=None, seed=42):
        self.meta     = processor_result
        self.fs       = fs
        self.duration = duration_sec
        self.seed     = seed
        self.factory  = KaspixSignalFactory(fs, duration_sec)
        self.recipe   = recipe if recipe else {'chirp': 1.0}

        if not self.meta['valid_spice']:
            raise ValueError('[Kaspix] Netlist inválido.')

        self.max_workers = multiprocessing.cpu_count()
        print(f'⚡ Kaspix AD8572 Generator: {self.max_workers} núcleos | fs={fs} Hz | dur={duration_sec}s | Seed:{seed}')

    def _sample_knobs_ad8572(self):
        """
        Muestrea knobs con distribución log-uniforme en sus rangos físicos.
        Log-uniform porque los componentes varían en décadas (10k→1M).
        """
        sampled = {}
        for knob_name, (lo, hi) in AD8572_KNOB_RANGES.items():
            if knob_name in self.meta['knobs_values']:
                # Log-uniforme: igual densidad por década
                log_val = np.random.uniform(np.log10(lo), np.log10(hi))
                sampled[knob_name] = 10 ** log_val
        return sampled

    def build_dataset(self, n_samples=100, save_path='kaspix_ad8572_dataset.pt'):
        set_global_seed(self.seed)
        print(f'--- Preparando {n_samples} tareas ---')

        tasks = []
        pre_generated_signals = {}

        for i in range(n_samples):
            sig_in, sig_type = self.factory.get_signal(self.recipe)
            knobs = self._sample_knobs_ad8572()

            pre_generated_signals[i] = {
                'audio_in': sig_in.astype(np.float32),
                'signal_type': sig_type
            }

            task = {
                'id': i,
                'file_path':     self.meta['file_path'],
                'input_source':  self.meta['input_source'],
                'output_target': self.meta['output_target'],
                'fs':       self.fs,
                'duration': self.duration,
                'input_signal': sig_in,
                'knob_config':  knobs
            }
            tasks.append(task)

        print(' Procesando en paralelo...')
        with concurrent.futures.ProcessPoolExecutor(max_workers=self.max_workers) as executor:
            results_unsorted = list(tqdm(
                executor.map(_simulation_worker_ad8572, tasks),
                total=n_samples,
                desc='Simulando AD8572'
            ))

        results_sorted = sorted(results_unsorted, key=lambda x: x['id'])

        dataset_x, dataset_y = [], []
        success_count, error_count = 0, 0

        for res in results_sorted:
            idx = res['id']
            if res['status'] == 'ok':
                input_meta  = pre_generated_signals[idx]
                audio_in    = input_meta['audio_in']
                raw_target  = res['y']

                # Limpieza y alineación
                audio_fix, target_fix, _ = clean_and_align(audio_in, raw_target)

                # Normalización Z-Score
                audio_norm  = (audio_fix  - np.mean(audio_fix))  / (np.std(audio_fix)  + 1e-8)
                target_norm = (target_fix - np.mean(target_fix)) / (np.std(target_fix) + 1e-8)

                x_entry = {
                    'audio_in':    audio_norm.astype(np.float32),
                    'signal_type': input_meta['signal_type'],
                    'knobs':       res['x_meta']['knobs'],
                    'knob_names':  res['x_meta']['knob_names'],
                    'circuit_id':  0,  # AD8572 inverting amp = ID 0
                    'netlist_origin': 'ad8572_inverting_amp.cir'
                }
                dataset_x.append(x_entry)
                dataset_y.append(target_norm.astype(np.float32))
                success_count += 1
            else:
                error_count += 1
                if error_count <= 5:
                    print(f'   Error muestra {idx}: {res["msg"]}')

        print(f'\n Resultados: {success_count} OK | {error_count} errores')

        if success_count == 0:
            print(' FALLO TOTAL — Revisar netlist y configuración.')
            return [], []

        print(f'Guardando {success_count} muestras en {save_path}...')
        torch.save({
            'x': dataset_x,
            'y': dataset_y,
            'metadata': {
                'fs':          self.fs,
                'duration':    self.duration,
                'recipe':      self.recipe,
                'knob_ranges': AD8572_KNOB_RANGES,
                'circuit_mapping': {'ad8572_inverting_amp.cir': 0},
                'component':   'AD8572',
                'topology':    'inverting_amplifier',
                'n_samples_total': success_count
            }
        }, save_path)

        print(f' Dataset guardado: {save_path}')
        return dataset_x, dataset_y


print(' KaspixParallelGeneratorAD8572 definido')

## Validación Rápida — Test de 3 muestras antes de producción

In [ ]:
# VALIDACIÓN RÁPIDA (3 muestras)
from scipy.signal import welch

FS_VAL       = 500_000   # 500 kHz — cubre GBW del AD8572 (1.5 MHz)
DURATION_VAL = 0.002     # 2 ms para test rápido

proc_val = NetlistProcessor(NETLIST_PATH)
meta_val = proc_val.analyze()

factory_val  = KaspixSignalFactory(FS_VAL, DURATION_VAL)
audio_in_val, sig_name = factory_val.get_signal({'chirp': 0.5, 'pink_noise': 0.5})

# Knobs nominales (ganancia -10x)
test_knobs = {'Rf': 100e3, 'Rg': 10e3, 'Rin': 1e3, 'Cf': 100e-12, 'Cin': 1e-9}

task_val = {
    'id': 0,
    'file_path':     NETLIST_PATH,
    'input_source':  meta_val['input_source'],
    'output_target': meta_val['output_target'],
    'fs':       FS_VAL,
    'duration': DURATION_VAL,
    'input_signal': audio_in_val,
    'knob_config':  test_knobs
}

print(f' Ejecutando test de validación (señal: {sig_name}, ganancia: -{test_knobs["Rf"]/test_knobs["Rg"]:.0f}x)...')
result_val = _simulation_worker_ad8572(task_val)
print(f'   Status: {result_val["status"]}')

if result_val['status'] == 'ok':
    audio_fix, target_fix, lag = clean_and_align(audio_in_val, result_val['y'])
    print(f'   Lag detectado: {lag} muestras')
    print(f'   RMS entrada: {np.std(audio_fix):.4f} V')
    print(f'   RMS salida:  {np.std(target_fix):.4f} V')
    print(f'   Ganancia RMS: {np.std(target_fix)/np.std(audio_fix):.2f}x (esperado ~{test_knobs["Rf"]/test_knobs["Rg"]:.0f}x)')

    # Visualización
    fig, axes = plt.subplots(1, 2, figsize=(15, 4))
    fig.suptitle('AD8572 — Validación Netlist NGSpice', fontsize=13, fontweight='bold')

    # Dominio tiempo
    n_plot = min(2000, len(audio_fix))
    t_plot = np.linspace(0, DURATION_VAL, len(audio_fix))[:n_plot] * 1e3  # ms
    axes[0].plot(t_plot, audio_fix[:n_plot], label='Entrada', alpha=0.8, linewidth=0.8)
    axes[0].plot(t_plot, target_fix[:n_plot], label='Salida (AD8572)', alpha=0.8, linewidth=0.8, linestyle='--')
    axes[0].set_xlabel('Tiempo [ms]')
    axes[0].set_ylabel('Voltaje [V]')
    axes[0].set_title(f'Dominio Tiempo | Señal: {sig_name}')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # Respuesta en frecuencia (Bode estimado)
    f_in, p_in   = welch(audio_fix,  FS_VAL, nperseg=512)
    f_out, p_out = welch(target_fix, FS_VAL, nperseg=512)
    mag = 10 * np.log10(p_out / (p_in + 1e-20))
    # Filtrar NaN/Inf
    valid = np.isfinite(mag) & (f_out > 0)
    axes[1].semilogx(f_out[valid], mag[valid], color='royalblue', linewidth=1.2)
    axes[1].axhline(y=20*np.log10(test_knobs['Rf']/test_knobs['Rg']),
                    color='red', linestyle='--', alpha=0.7,
                    label=f'Ganancia teórica ({20*np.log10(test_knobs["Rf"]/test_knobs["Rg"]):.1f} dB)')
    axes[1].axvline(x=1.5e6, color='orange', linestyle=':', alpha=0.7, label='GBW AD8572 (1.5 MHz)')
    axes[1].set_xlabel('Frecuencia [Hz]')
    axes[1].set_ylabel('Ganancia [dB]')
    axes[1].set_title('Respuesta en Frecuencia estimada (Welch)')
    axes[1].set_xlim([10, FS_VAL/2])
    axes[1].set_ylim([-40, 30])
    axes[1].legend()
    axes[1].grid(True, which='both', alpha=0.3)

    plt.tight_layout()
    plt.show()
    print(' Validación exitosa — Netlist funciona correctamente')
else:
    print(f' Validación fallida: {result_val["msg"]}')
    print('   → Revisa el netlist o reduce la complejidad del subcircuito')

## Generación de Dataset de Producción

**Configuración recomendada para el AD8572:**
- `fs = 500 kHz` → cubre el GBW del AD8572 (1.5 MHz) con margen
- `duration = 0.04 s` → 40 ms por muestra → 20,000 puntos por muestra
- `n_samples = 20,000` → dataset de producción

**Recipe de señales AD8572** (orientado a aplicaciones de sensor/precision):
- 30% chirp (barrido frecuencial — captura GBW)
- 25% pink_noise (cobertura espectral)
- 20% step_sequence (respuesta transitoria — captura SR)
- 15% sensor_ramp (caso de uso típico: puente de Wheatstone)
- 10% multitone (intermodulación)

In [ ]:
# BLOQUE PRINCIPAL DE PRODUCCIÓN PARA AD8572
if __name__ == '__main__':

    # --- CONFIGURACIÓN ---
    FS             = 500_000   # 500 kHz (cubre GBW del AD8572)
    DURATION       = 0.04      # 40 ms por muestra
    N_SAMPLES      = 2000    # Muestras totales
    SEED           = 42
    NETLIST        = NETLIST_PATH
    DATA_DIR       = 'datasets_ad8572'
    OUTPUT_DATASET = 'kaspix_ad8572_dataset.pt'

    # Opcional: montar Drive (Colab)
    # from google.colab import drive
    # drive.mount('/content/drive')
    # DRIVE_DEST = '/content/drive/MyDrive/kaspix/datasets/'

    os.makedirs(DATA_DIR, exist_ok=True)

    # Recipe específico para AD8572 (amplificador de precisión)
    RECIPE_MIX = {
        'chirp':        0.30,  # Barrido → captura GBW y rolloff
        'pink_noise':   0.25,  # Cobertura espectral completa
        'step_sequence':0.20,  # Transitorio → captura Slew Rate
        'sensor_ramp':  0.15,  # Caso de uso real (sensores)
        'multitone':    0.10,  # Intermodulación y linealidad
    }

    print(' KASPIX OMNI-PIPELINE — GEMELO DIGITAL AD8572')
    print(f'   Componente : Analog Devices AD8572')
    print(f'   Topología  : Amplificador Inversor de Precisión')
    print(f'   fs         : {FS/1e3:.0f} kHz')
    print(f'   Duración   : {DURATION*1e3:.0f} ms/muestra')
    print(f'   N muestras : {N_SAMPLES:,}')
    print(f'   Netlist    : {NETLIST}')
    print(f'   Knob Ranges: {AD8572_KNOB_RANGES}')
    print()

    proc = NetlistProcessor(NETLIST)
    meta = proc.analyze()

    if not meta['valid_spice']:
        print(' ERROR: Netlist inválido')
    else:
        generator = KaspixParallelGeneratorAD8572(
            processor_result=meta,
            fs=FS,
            duration_sec=DURATION,
            recipe=RECIPE_MIX,
            seed=SEED
        )

        save_path = os.path.join(DATA_DIR, OUTPUT_DATASET)

        dataset_x, dataset_y = generator.build_dataset(
            n_samples=N_SAMPLES,
            save_path=save_path
        )

        # --- Mover a Drive (descomentar si usas Colab + Drive) ---
        # if os.path.exists(save_path):
        #     os.makedirs(DRIVE_DEST, exist_ok=True)
        #     shutil.copy(save_path, os.path.join(DRIVE_DEST, OUTPUT_DATASET))
        #     print(f' Dataset copiado a Drive: {DRIVE_DEST}')

        if dataset_x:
            print(f'\n RESUMEN FINAL:')
            print(f'   Muestras generadas  : {len(dataset_x):,}')
            print(f'   Longitud por muestra: {len(dataset_x[0]["audio_in"]):,} puntos')
            print(f'   Knobs por muestra   : {len(dataset_x[0]["knobs"])} ({dataset_x[0]["knob_names"]})')
            print(f'   Dataset guardado en : {save_path}')

## Inspección del Dataset Generado

In [ ]:
# INSPECCIÓN Y ESTADÍSTICAS DEL DATASET
import torch
import numpy as np
import matplotlib.pyplot as plt

DATASET_PATH = os.path.join('datasets_ad8572', 'kaspix_ad8572_dataset.pt')

if os.path.exists(DATASET_PATH):
    data = torch.load(DATASET_PATH, weights_only=False)
    X    = data['x']
    Y    = data['y']
    meta = data['metadata']

    print('=' * 55)
    print('  KASPIX DATASET — AD8572 Inverting Amplifier')
    print('=' * 55)
    print(f'  Muestras totales    : {len(X):,}')
    print(f'  Longitud señal (pts): {len(X[0]["audio_in"]):,}')
    print(f'  Sample rate         : {meta["fs"]/1e3:.0f} kHz')
    print(f'  Duración muestra    : {meta["duration"]*1e3:.0f} ms')
    print(f'  Knobs               : {X[0]["knob_names"]}')
    print(f'  Componente          : {meta["component"]}')
    print(f'  Topología           : {meta["topology"]}')
    print()

    # Estadísticas de Knobs
    knob_names = X[0]['knob_names']
    all_knobs  = np.array([s['knobs'] for s in X])

    print('  Distribución de Knobs (log10):')
    for i, kname in enumerate(knob_names):
        vals = all_knobs[:, i]
        lo, hi = AD8572_KNOB_RANGES.get(kname, (vals.min(), vals.max()))
        print(f'    {kname:6s}: [{vals.min():.2e} – {vals.max():.2e}] '
              f'| Rango objetivo: [{lo:.2e} – {hi:.2e}]')

    print()

    # Distribución de tipos de señal
    sig_types = [s['signal_type'] for s in X]
    from collections import Counter
    cnt = Counter(sig_types)
    print('  Distribución de señales de entrada:')
    for stype, n in sorted(cnt.items(), key=lambda x: -x[1]):
        print(f'    {stype:20s}: {n:5d} ({100*n/len(X):.1f}%)')

    print()

    # Plot de ejemplo
    fig, axes = plt.subplots(2, 3, figsize=(16, 8))
    fig.suptitle('AD8572 Dataset — Muestras de Ejemplo', fontsize=13, fontweight='bold')

    indices = np.random.choice(len(X), 6, replace=False)
    for i, idx in enumerate(indices):
        ax  = axes[i // 3, i % 3]
        xi  = X[idx]
        yi  = Y[idx]
        n_p = min(1000, len(xi['audio_in']))
        ax.plot(xi['audio_in'][:n_p], label='Input', alpha=0.8, linewidth=0.8)
        ax.plot(yi[:n_p], label='Output', alpha=0.8, linewidth=0.8, linestyle='--')
        gain_rms = np.std(yi) / (np.std(xi['audio_in']) + 1e-8)
        knob_str = ', '.join([f'{k}={v:.2e}' for k, v in
                              zip(xi['knob_names'], xi['knobs'])])
        ax.set_title(f'#{idx} {xi["signal_type"]} | Gain≈{gain_rms:.1f}', fontsize=8)
        ax.legend(fontsize=7)
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
    print(' Inspección completa')

else:
    print(f'  Dataset no encontrado en: {DATASET_PATH}')
    print('  → Ejecuta primero la celda de producción.')